# Evaluate UNet Segmentation with Skeleton Metrics

Runs `segmentation_skeleton_metrics.evaluate.evaluate(...)` on the same brain/segmentation pair used by `load_skeletons.ipynb`. Outputs a CSV report and a text summary in `metrics_out/`.

**Note:** This re-reads the SWCs from GCS (~15 min) because the metrics package builds its own labeled-graph representation that differs from `BrainDataset`. The `dataset_cache_*.pkl` does not help here.

### Imports

In [1]:
import os
import sys

from segmentation_skeleton_metrics.evaluate import evaluate
from segmentation_skeleton_metrics.utils.img_util import TensorStoreImage

# Shared dataset-config helper (segmentation-id lookup).
sys.path.insert(0, "../scripts")
from dataset_config import get_segmentation_id

os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = "../configs/zihan_gcs_token.json"
os.environ["AWS_EC2_METADATA_DISABLED"] = "true"

### Paths

In [2]:
# Match load_skeletons.ipynb so we evaluate the same UNet checkpoint:
# segmentation id is looked up from configs/segmentation_datasets.rtf by brain_id.
brain_id = "794491"
segmentation_id = get_segmentation_id(brain_id)
print(f"brain_id={brain_id} -> segmentation_id={segmentation_id}")

gt_path = f"gs://allen-nd-goog/ground_truth_tracings/{brain_id}/voxel"
fragments_path = f"gs://allen-nd-goog/from_google/{brain_id}/whole_brain/{segmentation_id}/swcs"
segmentation_path = f"gs://allen-nd-goog/from_google/{brain_id}/whole_brain/{segmentation_id}/"

output_dir = f"../metrics_out/{brain_id}/{segmentation_id}"
os.makedirs(output_dir, exist_ok=True)

brain_id=794491 -> segmentation_id=mean40.stddev105.mask.136168199.no_omitted_20k.ffn.mt_0.1


### Run Evaluation

Three-step pipeline run by `evaluate(...)`:
1. **Label graphs** -- read GT SWCs and label each node with the UNet segment ID it falls inside (uses `segmentation`).
2. **Detect errors** -- per edge, classify as correct / split / omit / merge by comparing endpoint labels.
3. **Compute metrics** -- # Splits, # Merges, % Split/Omit/Merged Edges, ERL, Normalized ERL, Edge Accuracy, Split Rate, Merge Rate.

Saved artifacts: `results.csv` (per-GT-SWC), `results_overview.txt` (averaged + totals).

In [ ]:
# anisotropy: same (x, y, z) microns/voxel as load_skeletons.ipynb. Not
#   applied to the GT SWCs because gt_path ends in /voxel (already in voxel
#   coords); use_anisotropy=False reflects that.
anisotropy = (0.748, 0.748, 1.0)

# swap_axes=True: SWC voxel coords are stored (x, y, z); after the
# `from_google` transpose the segmentation volume's last 3 dims are
# (z, y, x). Setting swap_axes adds an additional transpose so the image's
# axes line up with the SWC's (x, y, z) ordering. Without this you get
# IndexError: OUT_OF_RANGE on the first GT skeleton labeling pass.
segmentation = TensorStoreImage(segmentation_path, swap_axes=True)

evaluate(
    gt_path,
    segmentation,
    output_dir,
    anisotropy=anisotropy,
    fragments_path=fragments_path,
    use_anisotropy=False,
    save_merges=True,
    save_fragments=False,
    verbose=True,
)

I0610 10:48:13.225280 3076689 google_auth_provider.cc:149] Using credentials at ../configs/zihan_gcs_token.json
I0610 10:48:13.226861 3076689 google_auth_provider.cc:165] Using ServiceAccount AuthProvider



(1) Load Ground Truth


Label Graphs: 100%|██████████| 9/9 [01:54<00:00, 12.77s/it]



(2) Load Fragments


Build Graphs:  74%|███████▍  | 522154/704080 [14:40<04:11, 722.58it/s]  

### Inspect Results

In [ ]:
import pandas as pd

results = pd.read_csv(os.path.join(output_dir, "results.csv"), index_col=0)
print(f"Per-SWC results ({len(results)} ground-truth skeletons):")
results

In [ ]:
with open(os.path.join(output_dir, "results_overview.txt")) as f:
    print(f.read())